# Week 2 Day 1 — Multi-Bot Conversations

**Author:** slavapa

Exercises from `week2/day1.ipynb`:
1. **2-way chat** with swapped personalities (optimist vs pessimist), plus prints that show how `messages` is built
2. **3-way chat** (GPT + Claude + Gemini) using one system prompt + one user prompt that embeds the full transcript
3. **Bonus:** swap Gemini for local Ollama (`llama3.2`)

Theme: three voices arguing about whether AI can help teach Kabbalah (ties to the course exercise sites).

**Keys needed:** `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `GOOGLE_API_KEY` in `.env`. Ollama optional for the bonus.

In [ ]:
import os

from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

In [ ]:
load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")

for name, key, n in [
    ("OpenAI", openai_api_key, 8),
    ("Anthropic", anthropic_api_key, 7),
    ("Google", google_api_key, 8),
]:
    if key:
        print(f"{name} key begins {key[:n]}")
    else:
        print(f"{name} key NOT SET")

In [ ]:
openai = OpenAI()
anthropic = OpenAI(
    api_key=anthropic_api_key,
    base_url="https://api.anthropic.com/v1/",
)
gemini = OpenAI(
    api_key=google_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)
ollama = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")

# Cheap / fast models — swap freely
GPT_MODEL = "gpt-4.1-mini"
CLAUDE_MODEL = "claude-haiku-4-5"
GEMINI_MODEL = "gemini-3.5-flash-lite"
OLLAMA_MODEL = "llama3.2"

## Part 1 — Two-way conversation (optimist vs pessimist)

Same stitching pattern as the lab: for each model, **its own lines are `assistant`** and the other bot's lines are `user`.

Set `DEBUG = True` to print the exact `messages` list sent on each call.

In [ ]:
DEBUG = True

optimist_system = (
    "You are Sunny, a relentlessly optimistic chatbot. "
    "You find the bright side of everything, cheer the other person on, "
    "and keep the conversation light. Keep replies to 2-3 short sentences."
)

pessimist_system = (
    "You are Gloomy, a witty pessimist chatbot. "
    "You poke holes in optimism, expect the worst, and sigh a lot — "
    "but stay playful, not cruel. Keep replies to 2-3 short sentences."
)

optimist_messages = ["Hi! Isn't it a wonderful day to learn something new?"]
pessimist_messages = ["Sure. Until something inevitably goes wrong."]


def _debug_messages(label, messages):
    if not DEBUG:
        return
    print(f"\n===== {label} messages ({len(messages)} items) =====")
    for m in messages:
        role = m["role"]
        content = m["content"].replace("\n", " ")[:120]
        print(f"  [{role}] {content}")


def call_optimist():
    """GPT as Sunny. Own history = assistant; Gloomy = user."""
    messages = [{"role": "system", "content": optimist_system}]
    for sunny, gloomy in zip(optimist_messages, pessimist_messages):
        messages.append({"role": "assistant", "content": sunny})
        messages.append({"role": "user", "content": gloomy})
    _debug_messages("Sunny / GPT", messages)
    response = openai.chat.completions.create(model=GPT_MODEL, messages=messages)
    return response.choices[0].message.content


def call_pessimist():
    """Claude as Gloomy. Own history = assistant; Sunny = user."""
    messages = [{"role": "system", "content": pessimist_system}]
    for sunny, gloomy in zip(optimist_messages, pessimist_messages):
        messages.append({"role": "user", "content": sunny})
        messages.append({"role": "assistant", "content": gloomy})
    messages.append({"role": "user", "content": optimist_messages[-1]})
    _debug_messages("Gloomy / Claude", messages)
    response = anthropic.chat.completions.create(model=CLAUDE_MODEL, messages=messages)
    return response.choices[0].message.content

In [ ]:
TURNS = 4

display(Markdown(f"### Sunny (GPT):\n{optimist_messages[0]}\n"))
display(Markdown(f"### Gloomy (Claude):\n{pessimist_messages[0]}\n"))

for i in range(TURNS):
    sunny_next = call_optimist()
    display(Markdown(f"### Sunny (GPT):\n{sunny_next}\n"))
    optimist_messages.append(sunny_next)

    gloomy_next = call_pessimist()
    display(Markdown(f"### Gloomy (Claude):\n{gloomy_next}\n"))
    pessimist_messages.append(gloomy_next)

## Part 2 — Three-way conversation (single user-prompt history)

With three speakers, `user` / `assistant` role alternation gets awkward. Instead:

- **System prompt** = persona + who else is in the room
- **User prompt** = full transcript so far + "say your next line as {Name}"

| Name | Model | Persona |
|------|-------|---------|
| Alex | GPT | Snarky skeptic |
| Blake | Claude | Polite bridge-builder |
| Charlie | Gemini | Enthusiastic visionary |

In [ ]:
PERSONAS = {
    "Alex": {
        "client": openai,
        "model": GPT_MODEL,
        "system": (
            "You are Alex, a snarky chatbot who challenges everything. "
            "You disagree with soft claims, poke holes, and stay witty. "
            "You are chatting with Blake and Charlie. "
            "Reply in 2-3 short sentences as Alex only — no name prefix."
        ),
    },
    "Blake": {
        "client": anthropic,
        "model": CLAUDE_MODEL,
        "system": (
            "You are Blake, a polite, courteous chatbot. "
            "You look for common ground and calm tension when others get sharp. "
            "You are chatting with Alex and Charlie. "
            "Reply in 2-3 short sentences as Blake only — no name prefix."
        ),
    },
    "Charlie": {
        "client": gemini,
        "model": GEMINI_MODEL,
        "system": (
            "You are Charlie, an enthusiastic visionary chatbot. "
            "You love big ideas, metaphors, and future possibilities. "
            "You are chatting with Alex and Blake. "
            "Reply in 2-3 short sentences as Charlie only — no name prefix."
        ),
    },
}

ORDER = ["Alex", "Blake", "Charlie"]


def format_transcript(conversation):
    if not conversation:
        return "(nothing yet — you open the discussion)"
    return "\n".join(f"{name}: {text}" for name, text in conversation)


def call_speaker(name, conversation, debug=False):
    persona = PERSONAS[name]
    others = [n for n in ORDER if n != name]
    user_prompt = (
        f"You are {name}, in conversation with {' and '.join(others)}.\n"
        f"The conversation so far is as follows:\n"
        f"{format_transcript(conversation)}\n\n"
        f"Now respond with what you would like to say next, as {name}. "
        f"Do not include your name or any prefix — only the spoken line."
    )
    messages = [
        {"role": "system", "content": persona["system"]},
        {"role": "user", "content": user_prompt},
    ]
    if debug:
        print(f"\n===== call {name} =====")
        print(user_prompt[:400], "..." if len(user_prompt) > 400 else "")

    response = persona["client"].chat.completions.create(
        model=persona["model"],
        messages=messages,
    )
    return response.choices[0].message.content.strip()


def run_three_way(topic, rounds=3, debug=False):
    conversation = []
    # Seed: Alex opens with the topic framing
    opener = (
        f"Alright, hot take: {topic} "
        "Blake will try to make peace; Charlie will get mystical. Prove me wrong."
    )
    conversation.append(("Alex", opener))
    display(Markdown(f"### Alex (GPT):\n{opener}\n"))

    for r in range(rounds):
        for name in ORDER:
            # Skip Alex's first turn of round 0 — already seeded
            if r == 0 and name == "Alex":
                continue
            line = call_speaker(name, conversation, debug=debug)
            conversation.append((name, line))
            model_tag = {"Alex": "GPT", "Blake": "Claude", "Charlie": "Gemini"}[name]
            display(Markdown(f"### {name} ({model_tag}):\n{line}\n"))

    return conversation

In [ ]:
TOPIC = (
    "AI can meaningfully help people learn Kabbalah — "
    "sites like kabbalahmedia.info and michaellaitman.com are just the start."
)

three_way = run_three_way(TOPIC, rounds=3, debug=True)

## Part 3 (bonus) — Replace Gemini with Ollama

Same 3-way pattern; Charlie runs on local `llama3.2`. Requires Ollama at `http://localhost:11434/`.

In [ ]:
import requests

try:
    ollama_ok = requests.get("http://localhost:11434", timeout=2).ok
except Exception:
    ollama_ok = False

print("Ollama available:" if ollama_ok else "Ollama NOT available — start it and pull llama3.2", ollama_ok)

if ollama_ok:
    PERSONAS["Charlie"] = {
        "client": ollama,
        "model": OLLAMA_MODEL,
        "system": (
            "You are Charlie, an enthusiastic visionary chatbot running locally. "
            "You love big ideas and keep replies to 2-3 short sentences. "
            "You are chatting with Alex and Blake. "
            "Reply as Charlie only — no name prefix."
        ),
    }

    TOPIC_OLLAMA = (
        "A local open-source model can join a Kabbalah study circle "
        "as usefully as a frontier cloud model."
    )
    ollama_chat = run_three_way(TOPIC_OLLAMA, rounds=2, debug=False)
else:
    ollama_chat = None

## Takeaways

1. **2-way chat** works with classic `user` / `assistant` history because each model only sees itself as assistant.
2. **3+ speakers** are clearer with a **single user prompt** that embeds the whole transcript — same idea whenever context is more complex than a simple back-and-forth (e.g. a tutor that needs prior session summaries).
3. Personas live in the **system prompt**; shared history lives in the **user prompt**.